In [11]:
!pip install sentence-transformers faiss-cpu flask -q

In [12]:
import numpy as np
import faiss
import re
from sentence_transformers import SentenceTransformer

In [13]:
data = [
    {"question": "What is admission deadline?",
     "answer": "Admission deadline is 30 August."},

    {"question": "What are the fee details?",
     "answer": "Fee is approximately 50,000 per semester."},

    {"question": "Which programs are offered?",
     "answer": "We offer BSCS, BBA, BSIT and AI programs."},

    {"question": "When do admissions open?",
     "answer": "Admissions open every year in August."},

    {"question": "Is entry test required?",
     "answer": "Yes, entry test is required for most programs."},

    {"question": "Is hostel available?",
     "answer": "Yes, hostel facility is available for students."}
]

In [14]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = text.strip()
    return text

In [15]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
questions = [preprocess(item["question"]) for item in data]
answers = [item["answer"] for item in data]

In [17]:
embeddings = model.encode(questions)
embeddings = np.array(embeddings).astype("float32")

In [18]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [19]:
def get_answer(query):
    query = preprocess(query)

    query_vec = model.encode([query])
    query_vec = np.array(query_vec).astype("float32")

    D, I = index.search(query_vec, k=1)

    return answers[I[0][0]]

In [20]:
print(get_answer("When is admission deadline?"))
print(get_answer("What programs are available?"))

Admission deadline is 30 August.
We offer BSCS, BBA, BSIT and AI programs.
